# A3 - Methods

In [23]:
import ast
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

metadata = pd.read_csv(
    "Data/Track A/observation_metadata.csv",
    low_memory=False
)

In [24]:
# Preprocessing settings from the course GCA-lite notebook
INCLUDE_TEACHER = False
MIN_WORDS = 1
STRIP_BRACKETS = True
W = 3
REPRESENTATION = "tfidf"


# Convert stored student-name lists into Python lists
def parse_student_names(value):
    if pd.isna(value):
        return []

    names = ast.literal_eval(value)

    return [
        str(name).strip()
        for name in names
    ] if isinstance(names, list) else []


metadata_gca = metadata.copy()

# Match metadata IDs (OBS-...) to transcript IDs (OBS_...)
metadata_gca["observation"] = (
    metadata_gca["obs_index"]
    .str.strip()
    .str.replace(r"^OBS-", "OBS_", regex=True)
)

metadata_gca["student_names"] = (
    metadata_gca["student_pseudonyms"]
    .apply(parse_student_names)
)

student_lookup = dict(
    zip(
        metadata_gca["observation"],
        metadata_gca["student_names"].apply(set)
    )
)

teacher_lookup = dict(
    zip(
        metadata_gca["observation"],
        metadata_gca["teacher_pseudonym"].fillna("").str.strip()
    )
)

print("Student and teacher lookups ready.")

Student and teacher lookups ready.


In [25]:
# Keep transcript rows in their original conversational order
df_gca = (
    observations
    .sort_values(["observation", "line_order"], kind="stable")
    .copy()
    .reset_index(drop=True)
)

df_gca["speaker"] = (
    df_gca["speaker"].fillna("").astype(str).str.strip()
)

# Identify contiguous segment runs BEFORE filtering rows.
# Later similarity comparisons will not cross these boundaries.
segment = df_gca["segment"].fillna("__missing__").astype(str)

new_segment = (
    df_gca["observation"].ne(df_gca["observation"].shift())
    | segment.ne(segment.shift())
)

df_gca["segment_run"] = new_segment.cumsum()

# Unique IDs let us trace results back to the original transcript.
df_gca["row_id"] = (
    df_gca["observation"].astype(str)
    + ":"
    + df_gca["line_order"].astype(str)
)

assert df_gca["row_id"].is_unique, "Check duplicate transcript row IDs."

# Use metadata to distinguish focal students, teachers, and other speakers.
df_gca["speaker_role"] = [
    "Student"
    if speaker in student_lookup.get(obs, set())
    else "Teacher"
    if speaker == teacher_lookup.get(obs, "") and speaker != ""
    else "Other/unknown"
    for obs, speaker in zip(
        df_gca["observation"],
        df_gca["speaker"]
    )
]

# Clean the original-language text.
df_gca["text"] = df_gca["dialogue"].fillna("").astype(str)

if STRIP_BRACKETS:
    df_gca["text"] = df_gca["text"].str.replace(
        r"\[.*?\]", " ", regex=True
    )

df_gca["text"] = (
    df_gca["text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Same word-count rule as the course Colab.
df_gca["n_words"] = df_gca["text"].str.split().str.len()

# Record exclusions before filtering.
print("Rows by speaker role:")
print(df_gca["speaker_role"].value_counts())

allowed_roles = (
    ["Student", "Teacher"]
    if INCLUDE_TEACHER
    else ["Student"]
)

df_gca = (
    df_gca[
        df_gca["speaker_role"].isin(allowed_roles)
        & (df_gca["n_words"] >= MIN_WORDS)
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nRetained contributions:", len(df_gca))
print("Observations retained:", df_gca["observation"].nunique())

Rows by speaker role:
speaker_role
Student          128415
Teacher           17463
Other/unknown      1931
Name: count, dtype: int64

Retained contributions: 122180
Observations retained: 444


In [26]:
# Count contributions and words for each speaker within each observation.
participation = (
    df_gca
    .groupby(["observation", "speaker"], as_index=False)
    .agg(
        turns=("row_id", "size"),
        words=("n_words", "sum")
    )
)

# Group size: number of speakers with retained contributions,
# following the course Colab.
participation["k"] = (
    participation.groupby("observation")["speaker"]
    .transform("nunique")
)

participation["turn_share"] = (
    participation["turns"]
    / participation.groupby("observation")["turns"].transform("sum")
)

participation["word_share"] = (
    participation["words"]
    / participation.groupby("observation")["words"].transform("sum")
)

# Original participation measure: share minus equal-share reference.
participation["participation_eq9"] = (
    participation["turn_share"] - 1 / participation["k"]
)

# Group-size-adjusted measure suggested in the paper's footnote.
participation["participation_fn1"] = (
    participation["k"] * participation["turn_share"] - 1
)

# Shares should sum to 1 within every observation.
assert np.allclose(
    participation.groupby("observation")["turn_share"].sum(),
    1
)

assert np.allclose(
    participation.groupby("observation")["word_share"].sum(),
    1
)

display(participation.head(12).round(3))

,observation,speaker,turns,words,k,turn_share,word_share,participation_eq9,participation_fn1
0,OBS_25-0155,Edgardo,40,263,2,0.615,0.668,0.115,0.231
1,OBS_25-0155,Jenna,25,131,2,0.385,0.332,-0.115,-0.231
2,OBS_25-0156,Doyle,11,108,4,0.172,0.146,-0.078,-0.312
3,OBS_25-0156,Jason,19,200,4,0.297,0.271,0.047,0.188
4,OBS_25-0156,Karan,20,136,4,0.312,0.184,0.062,0.250
5,OBS_25-0156,Rebekah,14,294,4,0.219,0.398,-0.031,-0.125
6,OBS_25-0157,Don,117,581,4,0.307,0.270,0.057,0.228
7,OBS_25-0157,Ethan,54,357,4,0.142,0.166,-0.108,-0.433
8,OBS_25-0157,Lindsay,57,286,4,0.150,0.133,-0.100,-0.402
9,OBS_25-0157,Sammy,153,928,4,0.402,0.431,0.152,0.606


In [27]:
# Load the multilingual model only if the embeddings option is selected.
embedding_model = None

if REPRESENTATION == "embeddings":
    from sentence_transformers import SentenceTransformer

    embedding_model = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )


def build_similarity(group):
    texts = group["text"].tolist()

    if REPRESENTATION == "tfidf":
        vectorizer = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b\w+\b"
        )

        # Handle observations containing only punctuation or symbols.
        if not any(re.search(r"(?u)\b\w+\b", text) for text in texts):
            return np.zeros((len(group), len(group)))

        X = vectorizer.fit_transform(texts)

    elif REPRESENTATION == "embeddings":
        X = embedding_model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )

    else:
        raise ValueError(
            "REPRESENTATION must be 'tfidf' or 'embeddings'."
        )

    return cosine_similarity(X)


# Check the function on one observation before running the full dataset.
example_observation = participation.loc[
    participation["k"] >= 2, "observation"
].iloc[0]

example_group = (
    df_gca[df_gca["observation"] == example_observation]
    .reset_index(drop=True)
)

example_SIM = build_similarity(example_group)

print("Example observation:", example_observation)
print("Retained contributions:", len(example_group))
print("Similarity matrix shape:", example_SIM.shape)

display(
    pd.DataFrame(
        example_SIM[:5, :5],
        index=example_group["row_id"].iloc[:5],
        columns=example_group["row_id"].iloc[:5]
    ).round(3)
)

Example observation: OBS_25-0155
Retained contributions: 65
Similarity matrix shape: (65, 65)


row_id,OBS_25-0155:2,OBS_25-0155:3,OBS_25-0155:4,OBS_25-0155:5,OBS_25-0155:6
row_id,,,,,
OBS_25-0155:2,1.0,0.000,0.0,0.000,0.0
OBS_25-0155:3,0.0,1.000,0.0,0.031,0.0
OBS_25-0155:4,0.0,0.000,1.0,0.000,0.0
OBS_25-0155:5,0.0,0.031,0.0,1.000,0.0
OBS_25-0155:6,0.0,0.000,0.0,0.000,1.0


In [28]:
def calculate_gca(group, SIM, window):
    group = group.reset_index(drop=True)

    speakers = sorted(group["speaker"].unique())
    speaker_list = group["speaker"].tolist()
    segment_runs = group["segment_run"].tolist()

    n = len(group)

    if window < 1:
        raise ValueError("Window must be at least 1.")

    if SIM.shape != (n, n):
        raise ValueError("Similarity matrix does not match the transcript.")

    # Average similarity for a particular speaker pair and lag.
    def cross_cohesion(initiator, responder, lag):
        values = [
            SIM[t - lag, t]
            for t in range(lag, n)
            if speaker_list[t - lag] == initiator
            and speaker_list[t] == responder
            and segment_runs[t - lag] == segment_runs[t]
        ]

        # Same missing-lag convention as the course Colab.
        return np.mean(values) if values else 0.0

    # Rows = responders; columns = initiators.
    R = pd.DataFrame(
        [
            [
                np.mean([
                    cross_cohesion(initiator, responder, lag)
                    for lag in range(1, window + 1)
                ])
                for initiator in speakers
            ]
            for responder in speakers
        ],
        index=speakers,
        columns=speakers
    )

    R.index.name = "responder"
    R.columns.name = "initiator"

    # Record available comparison pairs so zero scores can be interpreted.
    support = pd.DataFrame(
        0, index=speakers, columns=speakers
    )

    for t in range(n):
        for lag in range(1, min(window, t) + 1):
            previous = t - lag

            if segment_runs[previous] == segment_runs[t]:
                support.loc[
                    speaker_list[t], speaker_list[previous]
                ] += 1

    rows = []

    for speaker in speakers:
        others = [
            other for other in speakers
            if other != speaker
        ]

        rows.append({
            "speaker": speaker,

            # Row mean, excluding the diagonal.
            "overall_responsivity": (
                R.loc[speaker, others].mean()
                if others else np.nan
            ),

            # Column mean, excluding the diagonal.
            "social_impact": (
                R.loc[others, speaker].mean()
                if others else np.nan
            ),

            # Diagonal entry.
            "internal_cohesion": R.loc[speaker, speaker],

            "responsivity_pair_count": (
                int(support.loc[speaker, others].sum())
            ),

            "social_impact_pair_count": (
                int(support.loc[others, speaker].sum())
            ),

            "internal_cohesion_pair_count": (
                int(support.loc[speaker, speaker])
            )
        })

    return R, pd.DataFrame(rows)


# Apply the function to our example observation.
example_R, example_gca = calculate_gca(
    example_group,
    example_SIM,
    window=W
)

print("Responsivity matrix:")
display(example_R.round(3))

print("GCA measures:")
display(example_gca.round(3))

Responsivity matrix:


initiator,Edgardo,Jenna
responder,,
Edgardo,0.110,0.027
Jenna,0.033,0.058


GCA measures:


,speaker,overall_responsivity,social_impact,internal_cohesion,responsivity_pair_count,social_impact_pair_count,internal_cohesion_pair_count
0,Edgardo,0.027,0.033,0.110,25,17,78
1,Jenna,0.033,0.027,0.058,17,25,39


In [29]:
# Run GCA-lite separately within each observation.
all_gca_results = []
responsivity_matrices = {}
run_log = []

groups = df_gca.groupby("observation", sort=True)
total = groups.ngroups

for number, (observation_id, group) in enumerate(groups, start=1):
    group = group.reset_index(drop=True)

    SIM = build_similarity(group)

    R, group_gca = calculate_gca(
        group,
        SIM,
        window=W
    )

    group_gca["observation"] = observation_id
    all_gca_results.append(group_gca)

    responsivity_matrices[observation_id] = R

    run_log.append({
        "observation": observation_id,
        "retained_contributions": len(group),
        "active_speakers": group["speaker"].nunique(),
        "segments": group["segment_run"].nunique(),
        "zero_vector_contributions": int(
            np.isclose(np.diag(SIM), 0).sum()
        ),
        "status": (
            "completed"
            if group["speaker"].nunique() >= 2
            else "one_speaker_no_peer_comparison"
        )
    })

    if number % 50 == 0 or number == total:
        print(f"Processed {number}/{total} observations")

if not all_gca_results:
    raise ValueError("No eligible observations were available for GCA.")

gca_results = pd.concat(
    all_gca_results,
    ignore_index=True
)

# Attach participation measures calculated earlier.
gca_results = participation.merge(
    gca_results,
    on=["observation", "speaker"],
    how="left",
    validate="one_to_one"
)

gca_results["representation"] = REPRESENTATION
gca_results["window"] = W

# Include observations that had no retained contributions in the log.
run_log = pd.DataFrame(run_log)

all_observation_ids = (
    observations[["observation"]]
    .drop_duplicates()
)

run_log = all_observation_ids.merge(
    run_log,
    on="observation",
    how="left",
    validate="one_to_one"
)

run_log["status"] = run_log["status"].fillna(
    "excluded_no_eligible_contributions"
)

print("\nRun status:")
print(run_log["status"].value_counts())

print("\nSpeaker-observation records:", len(gca_results))

display(gca_results.head(12).round(3))

Processed 50/444 observations
Processed 100/444 observations
Processed 150/444 observations
Processed 200/444 observations
Processed 250/444 observations
Processed 300/444 observations
Processed 350/444 observations
Processed 400/444 observations
Processed 444/444 observations

Run status:
status
completed                             443
one_speaker_no_peer_comparison          1
excluded_no_eligible_contributions      1
Name: count, dtype: int64

Speaker-observation records: 1356


,observation,speaker,turns,words,k,turn_share,word_share,participation_eq9,participation_fn1,overall_responsivity,social_impact,internal_cohesion,responsivity_pair_count,social_impact_pair_count,internal_cohesion_pair_count,representation,window
0,OBS_25-0155,Edgardo,40,263,2,0.615,0.668,0.115,0.231,0.027,0.033,0.110,25,17,78,tfidf,3
1,OBS_25-0155,Jenna,25,131,2,0.385,0.332,-0.115,-0.231,0.033,0.027,0.058,17,25,39,tfidf,3
2,OBS_25-0156,Doyle,11,108,4,0.172,0.146,-0.078,-0.312,0.071,0.039,0.052,23,13,3,tfidf,3
3,OBS_25-0156,Jason,19,200,4,0.297,0.271,0.047,0.188,0.039,0.113,0.063,28,38,13,tfidf,3
4,OBS_25-0156,Karan,20,136,4,0.312,0.184,0.062,0.250,0.074,0.116,0.250,38,36,15,tfidf,3
5,OBS_25-0156,Rebekah,14,294,4,0.219,0.398,-0.031,-0.125,0.144,0.059,0.161,25,27,5,tfidf,3
6,OBS_25-0157,Don,117,581,4,0.307,0.270,0.057,0.228,0.105,0.089,0.109,217,228,117,tfidf,3
7,OBS_25-0157,Ethan,54,357,4,0.142,0.166,-0.108,-0.433,0.075,0.117,0.096,131,128,28,tfidf,3
8,OBS_25-0157,Lindsay,57,286,4,0.150,0.133,-0.100,-0.402,0.090,0.082,0.125,139,136,32,tfidf,3
9,OBS_25-0157,Sammy,153,928,4,0.402,0.431,0.152,0.606,0.097,0.078,0.183,273,268,170,tfidf,3


In [30]:
# Check that participation shares sum to 1 within each observation.
assert np.allclose(
    gca_results.groupby("observation")["turn_share"].sum(),
    1
)

assert np.allclose(
    gca_results.groupby("observation")["word_share"].sum(),
    1
)

# The group-size-adjusted participation scores should sum to 0.
assert np.allclose(
    gca_results.groupby("observation")["participation_fn1"].sum(),
    0,
    atol=1e-10
)

# Responsivity and social impact are row/column averages
# of the same off-diagonal matrix entries.
group_means = (
    gca_results
    .groupby("observation")[
        ["overall_responsivity", "social_impact"]
    ]
    .mean()
    .dropna()
)

assert np.allclose(
    group_means["overall_responsivity"],
    group_means["social_impact"]
)

metrics = [
    "participation_fn1",
    "overall_responsivity",
    "social_impact",
    "internal_cohesion"
]

print("Calculation checks passed.")

print("\nDescriptive statistics across speaker-observation records:")
display(gca_results[metrics].describe().round(3))

print("\nRecords with no available comparison pairs:")
display(
    pd.Series({
        "Responsivity": (
            gca_results["responsivity_pair_count"] == 0
        ).sum(),
        "Social impact": (
            gca_results["social_impact_pair_count"] == 0
        ).sum(),
        "Internal cohesion": (
            gca_results["internal_cohesion_pair_count"] == 0
        ).sum()
    }, name="record_count").to_frame()
)

Calculation checks passed.

Descriptive statistics across speaker-observation records:


,participation_fn1,overall_responsivity,social_impact,internal_cohesion
count,1356.000,1355.000,1355.000,1356.000
mean,-0.000,0.088,0.088,0.126
std,0.388,0.038,0.037,0.059
min,-0.982,0.000,0.000,0.000
25%,-0.243,0.065,0.064,0.091
50%,-0.002,0.084,0.084,0.120
75%,0.217,0.106,0.106,0.151
max,1.789,0.350,0.350,0.661



Records with no available comparison pairs:


,record_count
Responsivity,3
Social impact,3
Internal cohesion,14


In [31]:
def get_peer_pairs(group, SIM, window):
    group = group.reset_index(drop=True)
    rows = []

    for t in range(len(group)):
        for lag in range(1, min(window, t) + 1):
            previous = t - lag

            same_segment = (
                group.loc[previous, "segment_run"]
                == group.loc[t, "segment_run"]
            )

            different_speakers = (
                group.loc[previous, "speaker"]
                != group.loc[t, "speaker"]
            )

            if same_segment and different_speakers:
                rows.append({
                    "previous_index": previous,
                    "current_index": t,
                    "lag": lag,
                    "similarity": float(SIM[previous, t])
                })

    return pd.DataFrame(
        rows,
        columns=[
            "previous_index",
            "current_index",
            "lag",
            "similarity"
        ]
    )


def show_pairs(pairs, group):
    if pairs.empty:
        print("No eligible peer pairs.")
        return

    for pair in pairs.itertuples(index=False):
        earlier = group.iloc[pair.previous_index]
        later = group.iloc[pair.current_index]

        print(
            f"Similarity: {pair.similarity:.3f} | Lag: {pair.lag}"
        )

        print(
            f"Line {earlier['line_order']} | "
            f"{earlier['speaker']}: {earlier['text']}"
        )

        print(
            f"Line {later['line_order']} | "
            f"{later['speaker']}: {later['text']}\n"
        )


example_peer_pairs = get_peer_pairs(
    example_group,
    example_SIM,
    window=W
)

print("Observation:", example_observation)

print("\nFive most similar peer pairs:\n")
show_pairs(
    example_peer_pairs.nlargest(5, "similarity"),
    example_group
)

print("\nFive least similar adjacent peer pairs:\n")
show_pairs(
    example_peer_pairs[
        example_peer_pairs["lag"] == 1
    ].nsmallest(5, "similarity"),
    example_group
)

Observation: OBS_25-0155

Five most similar peer pairs:

Similarity: 0.507 | Lag: 1
Line 47 | Edgardo: ¿No hay otro?
Line 48 | Jenna: No.

Similarity: 0.282 | Lag: 1
Line 119 | Jenna: ¿Y ya?
Line 120 | Edgardo: Si ¿Y tú qué?

Similarity: 0.250 | Lag: 2
Line 55 | Jenna: Ya lo encontré.
Line 57 | Edgardo: Esto es lo que yo pienso.

Similarity: 0.192 | Lag: 1
Line 120 | Edgardo: Si ¿Y tú qué?
Line 121 | Jenna: Yo miro que son diferentes fracciones y la D solo tiene de enteros y son de más enteros o menos y hay muchos nueves.

Similarity: 0.113 | Lag: 3
Line 30 | Jenna: Más grande.
Line 39 | Edgardo: So I guess six más dos o six times two.


Five least similar adjacent peer pairs:

Similarity: 0.000 | Lag: 1
Line 2 | Edgardo: What I know about this is it is a number greater than one.
Line 3 | Jenna: Okay. Yo pienso que sería un poco imposible porque casi no se multiplican numeros así muy grandes con fracciones.

Similarity: 0.000 | Lag: 1
Line 3 | Jenna: Okay. Yo pienso que sería un poco i

In [33]:
def calculate_newness(group):
    group = group.reset_index(drop=True)

    result = group[
        ["observation", "row_id", "speaker", "segment"]
    ].copy()

    result["newness"] = np.nan
    result["has_prior_content"] = False

    texts = group["text"].tolist()

    if REPRESENTATION == "tfidf":
        if not any(re.search(r"(?u)\b\w+\b", text) for text in texts):
            return result

        vectorizer = TfidfVectorizer(
            lowercase=True,
            token_pattern=r"(?u)\b\w+\b"
        )

        X = vectorizer.fit_transform(texts)

    elif REPRESENTATION == "embeddings":
        X = embedding_model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )

    else:
        raise ValueError("Unknown representation.")

    # Reset the given-content space at each segment boundary.
    for _, segment_group in group.groupby("segment_run", sort=False):
        indices = segment_group.index.to_numpy()

        vectors = X[indices]

        if hasattr(vectors, "toarray"):
            vectors = vectors.toarray()

        vectors = np.asarray(vectors, dtype=float)

        # Rows of basis form an orthonormal representation
        # of all earlier nonzero contributions in this segment.
        basis = np.zeros((
            min(len(vectors), vectors.shape[1]),
            vectors.shape[1]
        ))

        rank = 0

        for position, vector in zip(indices, vectors):
            vector_norm = np.linalg.norm(vector)

            # No lexical/semantic vector: leave newness undefined.
            if vector_norm <= 1e-12:
                continue

            result.loc[position, "has_prior_content"] = rank > 0

            residual = vector.copy()

            if rank > 0:
                Q = basis[:rank]

                # Two passes reduce numerical error.
                for _ in range(2):
                    residual -= (Q @ residual) @ Q

            new_norm = np.linalg.norm(residual)

            if new_norm <= 1e-8 * vector_norm:
                residual[:] = 0.0
                new_norm = 0.0

            given_norm = np.linalg.norm(vector - residual)

            # GCA given–new proportion: ||new|| / (||new|| + ||given||).
            result.loc[position, "newness"] = (
                new_norm / (new_norm + given_norm)
            )

            if new_norm > 0 and rank < len(basis):
                basis[rank] = residual / new_norm
                rank += 1

    return result

In [34]:
newness_tables = []

for observation_id, group in df_gca.groupby("observation", sort=True):
    newness_tables.append(calculate_newness(group))

newness_turns = pd.concat(
    newness_tables,
    ignore_index=True
)

# Also report a mean excluding contributions with no prior content.
newness_turns["newness_with_context"] = (
    newness_turns["newness"]
    .where(newness_turns["has_prior_content"])
)

newness_summary = (
    newness_turns
    .groupby(["observation", "speaker"], as_index=False)
    .agg(
        newness=("newness", "mean"),
        newness_with_context=("newness_with_context", "mean"),
        newness_valid_contributions=("newness", "count")
    )
)

# Allow this cell to be rerun without creating duplicate columns.
new_columns = [
    "newness",
    "newness_with_context",
    "newness_valid_contributions"
]

gca_results = (
    gca_results
    .drop(columns=new_columns, errors="ignore")
    .merge(
        newness_summary,
        on=["observation", "speaker"],
        how="left",
        validate="one_to_one"
    )
)

assert newness_turns["row_id"].is_unique

valid_scores = newness_turns["newness"].dropna()
assert valid_scores.between(0, 1).all()

print("Newness calculation completed.")

display(
    gca_results[
        ["observation", "speaker"] + new_columns
    ].head(12).round(3)
)

display(
    gca_results[
        ["newness", "newness_with_context"]
    ].describe().round(3)
)

Newness calculation completed.


,observation,speaker,newness,newness_with_context,newness_valid_contributions
0,OBS_25-0155,Edgardo,0.805,0.790,40
1,OBS_25-0155,Jenna,0.873,0.856,25
2,OBS_25-0156,Doyle,0.780,0.732,11
3,OBS_25-0156,Jason,0.885,0.872,19
4,OBS_25-0156,Karan,0.662,0.644,20
5,OBS_25-0156,Rebekah,0.817,0.787,14
6,OBS_25-0157,Don,0.565,0.554,117
7,OBS_25-0157,Ethan,0.628,0.628,54
8,OBS_25-0157,Lindsay,0.459,0.459,57
9,OBS_25-0157,Sammy,0.442,0.431,153


,newness,newness_with_context
count,1356.000,1356.000
mean,0.546,0.536
std,0.143,0.138
min,0.000,0.000
25%,0.443,0.437
50%,0.544,0.536
75%,0.644,0.634
max,1.000,1.000


In [36]:
from pathlib import Path
import json

DATA_DIR = Path("Data/Track A")

# Keep generated results alongside local data.
OUTPUT_DIR = (
    DATA_DIR
    / "gca_outputs"
    / f"{REPRESENTATION}_w{W}"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Speaker-level participation and GCA measures.
gca_results.to_csv(
    OUTPUT_DIR / "gca_student_scores.csv",
    index=False,
    encoding="utf-8-sig"
)

# 2. Processing status for every observation.
run_log.to_csv(
    OUTPUT_DIR / "gca_run_log.csv",
    index=False,
    encoding="utf-8-sig"
)

# 3. Save each responsivity matrix in one combined, long-format table.
matrix_tables = []

for observation_id, R in responsivity_matrices.items():
    matrix_table = (
        R.stack()
        .rename("responsivity")
        .reset_index()
    )

    matrix_table.insert(
        0, "observation", observation_id
    )

    matrix_tables.append(matrix_table)

responsivity_long = pd.concat(
    matrix_tables,
    ignore_index=True
)

responsivity_long.to_csv(
    OUTPUT_DIR / "gca_responsivity_matrices.csv",
    index=False,
    encoding="utf-8-sig"
)

# 4. Record preprocessing and calculation decisions.
settings = {
    "representation": REPRESENTATION,
    "window": W,
    "include_teacher": INCLUDE_TEACHER,
    "min_words": MIN_WORDS,
    "strip_brackets": STRIP_BRACKETS,
    "text_language": "original; no translation",
    "contribution_unit": "one retained transcript row",
    "grouping": "separate calculations within each observation",
    "segment_boundary": "comparisons cannot cross segment runs",
    "lag_definition": "distance in retained contributions",
    "missing_speaker_lag_cells": 0.0,
    "participation_group_size": "speakers with retained contributions",
    "newness_method": "orthogonal projection in the selected vector space",
    "newness_history": "all earlier retained contributions within the same segment",
    "newness_first_valid_contribution": 1.0,
    "newness_zero_vectors": "undefined; excluded from means",
    "raw_observations": int(
        observations["observation"].nunique()
    ),
    "scored_observations": int(
        gca_results["observation"].nunique()
    ),
    "retained_contributions": int(len(df_gca))
}

with open(
    OUTPUT_DIR / "gca_settings.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(settings, file, indent=2, ensure_ascii=False)

print("Results saved to:", OUTPUT_DIR.resolve())

for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)

Results saved to: /Users/Apple/Desktop/EDUC 435/Data/Track A/gca_outputs/tfidf_w3
gca_responsivity_matrices.csv
gca_run_log.csv
gca_settings.json
gca_student_scores.csv
